In [1]:
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields

In [2]:
model = GmshDiscreteModel("SingleEdgeNotchRect_15degneg.msh")
writevtk(model,"SingleEdgeNotchRect_15degneg")

Info    : Reading 'SingleEdgeNotchRect_15degneg.msh'...
Info    : 20 entities
Info    : 3389 nodes
Info    : 6715 elements
Info    : Done reading 'SingleEdgeNotchRect_15degneg.msh'


3-element Vector{Vector{String}}:
 ["SingleEdgeNotchRect_15degneg_0.vtu"]
 ["SingleEdgeNotchRect_15degneg_1.vtu"]
 ["SingleEdgeNotchRect_15degneg_2.vtu"]

In [3]:
# sqrt(0.2*83160)
sqrt(114800*11700)

36649.14732978109

In [4]:
const ν = 0.21
const E = 33000 
const G = E/(2*(1+ν))

13636.363636363636

In [5]:
const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = G
const κ = λ + μ

23510.971786833856

In [6]:
D_x = 2 
D_y = 2 
G0 = 3e8
r = 0.016
τ = 1e-6
Gc = 1500  #σ_crit_ts
delT1= 1e-4 

0.0001

In [7]:
θ = -15
a_vec = VectorValue(cosd(θ), sind(θ))
A_Ten = a_vec ⊗ a_vec
A₁₁ = cosd(θ)
A₁₂ = -sind(θ)
A₂₁ = sind(θ)
A₂₂ =  cosd(θ)
T_Mat = TensorValue(A₁₁,A₂₁,A₁₂,A₂₂)
χ = 0.0 #1e5

0.0

In [8]:
σ(ε)= λ*tr(ε)*one(ε) + 2*μ*ε + χ * tr(ε) * A_Ten 

σ (generic function with 1 method)

In [9]:
degree = 2
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)

GenericMeasure()

In [10]:
labels = get_face_labeling(model)

FaceLabeling:
 0-faces: 3389
 1-faces: 10004
 2-faces: 6616
 tags: 5
 entities: 20

In [11]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

6616-element Gridap.Arrays.Table{Int32, Vector{Int32}, Vector{Int32}}:
 [238, 282, 2074]
 [233, 235, 3320]
 [1874, 2231, 3034]
 [276, 277, 3222]
 [2002, 2057, 3321]
 [1473, 1476, 2071]
 [197, 1920, 3278]
 [282, 1977, 2075]
 [1160, 1982, 2117]
 [1902, 2078, 2079]
 [452, 1859, 2081]
 [1160, 1982, 3346]
 [1854, 1891, 2137]
 ⋮
 [2446, 2938, 2939]
 [2453, 2454, 3307]
 [2454, 2939, 3307]
 [2458, 2939, 3307]
 [2936, 2937, 2938]
 [3109, 3110, 3111]
 [3109, 3174, 3177]
 [3173, 3174, 3175]
 [3173, 3334, 3335]
 [3200, 3263, 3267]
 [3261, 3262, 3264]
 [3262, 3263, 3265]

In [12]:
function σ_eq1(ε)
    εArray = get_array(T_Mat' ⋅ ε ⋅ T_Mat)
    εArray_Fibre = get_array(T_Mat' ⋅ (ε⋅(A_Ten)) ⋅ T_Mat)
    Λ, P = eigen(εArray)
    Λ_Fibre, P_Fibre = eigen(εArray_Fibre)
    # Positive part of strain
    Λpos = diagm(0 => max.(0, Λ))
    εPos = TensorValue(P * Λpos * P')
    Λpos_Fibre = diagm(0 => max.(0.0, Λ_Fibre))
    εPos_Fibre = TensorValue(P_Fibre * Λpos_Fibre * P_Fibre')

    # Components
    εxx, εyy, εxy = εPos[1,1], εPos[2,2], εPos[1,2]
    εxx_Fibre, εyy_Fibre, εxy_Fibre = εPos_Fibre[1,1], εPos_Fibre[2,2], εPos_Fibre[1,2]

    # --- Volumetric part split ---
    trε = tr(ε)
    

    # Split volumetric part between x and y
    vol_x = (trε >= 0) ? 0.5 * λ * (εxx^2 + εxx*εyy) : 0.0
    vol_y = (trε >= 0) ? 0.5 * λ * (εyy^2 + εxx*εyy) : 0.0
    vol_x_Fibre = (trε >= 0) ? 0.5 * χ * (εxx_Fibre^2 + εxx_Fibre*εyy_Fibre) : 0.0
    vol_y_Fibre = (trε >= 0) ? 0.5 * χ * (εyy_Fibre^2 + εxx_Fibre*εyy_Fibre) : 0.0

    # --- Deviatoric part split ---
    dev_x = μ * (εxx^2 + εxy^2)
    dev_y = μ * (εyy^2 + εxy^2)
    dev_x_Fibre = χ * (εxx_Fibre^2 + εxy_Fibre^2)
    dev_y_Fibre = χ * (εyy_Fibre^2 + εxy_Fibre^2)

    # Combine
    ψPos_x = 0.5 * (vol_x + 2*dev_x + vol_x_Fibre + 2*dev_x_Fibre)
    ψPos_y = 0.5 * (vol_y + 2*dev_y + vol_y_Fibre + 2*dev_y_Fibre)

 

    return sqrt(2*ψPos_x*E)
end

function σ_eq2(ε)
 εArray = get_array(T_Mat' ⋅ ε ⋅ T_Mat)
    εArray_Fibre = get_array(T_Mat' ⋅ (ε⋅(A_Ten)) ⋅ T_Mat)
    Λ, P = eigen(εArray)
    Λ_Fibre, P_Fibre = eigen(εArray_Fibre)
    # Positive part of strain
    Λpos = diagm(0 => max.(0, Λ))
    εPos = TensorValue(P * Λpos * P')
    Λpos_Fibre = diagm(0 => max.(0, Λ_Fibre))
    εPos_Fibre = TensorValue(P_Fibre * Λpos_Fibre * P_Fibre')

    # Components
    εxx, εyy, εxy = εPos[1,1], εPos[2,2], εPos[1,2]
    εxx_Fibre, εyy_Fibre, εxy_Fibre = εPos_Fibre[1,1], εPos_Fibre[2,2], εPos_Fibre[1,2]

    # --- Volumetric part split ---
    trε = tr(ε)
    

    # Split volumetric part between x and y
    vol_x = (trε >= 0) ? 0.5 * λ * (εxx^2 + εxx*εyy) : 0.0
    vol_y = (trε >= 0) ? 0.5 * λ * (εyy^2 + εxx*εyy) : 0.0
    vol_x_Fibre = (trε >= 0) ? 0.5 * χ  * (εxx_Fibre^2 + εxx_Fibre*εyy_Fibre) : 0.0
    vol_y_Fibre = (trε >= 0) ? 0.5 * χ  * (εyy_Fibre^2 + εxx_Fibre*εyy_Fibre) : 0.0

    # --- Deviatoric part split ---
    dev_x = μ * (εxx^2 + εxy^2)
    dev_y = μ * (εyy^2 + εxy^2)
    dev_x_Fibre = χ  * (εxx_Fibre^2 + εxy_Fibre^2)
    dev_y_Fibre = χ  * (εyy_Fibre^2 + εxy_Fibre^2)

    # Combine
    ψPos_x = 0.5 * (vol_x + 2*dev_x + vol_x_Fibre + dev_x_Fibre)
    ψPos_y = 0.5 * (vol_y + 2*dev_y + vol_y_Fibre + dev_y_Fibre)

    return sqrt(2*(ψPos_y)*E)
end

σ_eq2 (generic function with 1 method)

In [13]:
function G_kill(σ_eq_x,σ_eq_y)
G_kill_x = 0.5*G0.*((σ_eq_x /Gc- 1) + abs(σ_eq_x/Gc- 1))
G_kill_y = 0.5*G0.*((σ_eq_y /Gc- 1) + abs(σ_eq_y/Gc- 1))
    return G_kill_x, G_kill_y
end 

G_kill (generic function with 1 method)

In [14]:
function new_EnergyState(ψPlusPrev_in,t_s_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
         tPlus_out = t_s_in
        damaged = false
    else
        ψPlus_out = ψPlus_in
        tPlus_out = T
        damaged = true
    end
    damaged, ψPlus_out, tPlus_out   
end

new_EnergyState (generic function with 1 method)

In [15]:
function barw_closed_form(delT1,m_p_x,m_p_y,G_k_in_x,G_k_in_y,σ_p_x,σ_p_y)
    s_x = Cal_Var(σ_p_x,D_x,τ,G_k_in_x,delT1)
    s_y = Cal_Var(σ_p_y,D_y,τ,G_k_in_y,delT1)
    μ_x = Cal_mean(m_p_x,G_k_in_x,τ,delT1,D_x)
    μ_y = Cal_mean(m_p_y,G_k_in_y,τ,delT1,D_y)
    bar_w_nds_x = 1.0 ./ sqrt.(1.0 .+ s_x) .* (exp.( - ( μ_x .*μ_x ) ./ (2.0 * (1.0 .+  s_x))) )
    bar_w_nds_y = 1.0 ./ sqrt.(1.0 .+ s_y) .* (exp.( - ( μ_y .*μ_y ) ./ (2.0 * (1.0 .+  s_y))) )
    bar_w_nds =  bar_w_nds_x.*bar_w_nds_y 
    return FEFunction(Vphi, bar_w_nds.+1e-6), μ_x, s_x, μ_y, s_y
end


function Cal_mean(m_p,G_k_in,τ,delT1,D)
    g = -4 .* G_k_in .* D .* delT1 .* m_p
    H = -4 .* G_k_in .* D .* delT1 
        update_mask = 1 .+  τ .* H .>= 1e-2
        m_new= copy(m_p)

    if !any(update_mask)
        return min.(m_new,1000.0)
    end
    H_subset = H[update_mask]
    m_p_subset = m_p[update_mask]
    g_subset = g[update_mask]
   
   m_updated = (m_p_subset .- τ .* g_subset) ./ (1 .+ τ .* H_subset)
   m_new[update_mask] = min.(m_updated,1000.0)
    return m_new
end

function Cal_Var(σ_p,D,τ,G_k_in, delT1)
    H = -4 .* G_k_in .* D .* delT1 
    β = sqrt.(σ_p)./(τ*D)
    c = 1/(2*τ*D) .+ H ./ 2
    
    update_mask = (β.^2) .+ (8 .* c) .>= 1e-2

    σ_new = copy(σ_p)

    if !any(update_mask)
        return min.(σ_new,1000.0)
    end

    β_subset = β[update_mask]
    c_subset = c[update_mask]

    y1 = (-β_subset .+ sqrt.(β_subset.^2 .+ 8 .* c_subset)) ./ 2
    y2 = (-β_subset .- sqrt.(β_subset.^2 .+ 8 .* c_subset)) ./ 2
    y_subset = max.(y1, y2)
    
    σ_updated_values = 1 ./ (y_subset.^2)

    σ_new[update_mask] = min.(σ_updated_values,1000.0)
    return σ_new
end

Cal_Var (generic function with 1 method)

In [16]:
using NearestNeighbors
data = zeros(2,Nₙ)
data[1,:] =nodeCoordX
data[2,:] =nodeCoordY

points = data

balltree = BallTree(data)
idxs = inrange(balltree, points, r, true)

3389-element Vector{Vector{Int64}}:
 [1]
 [2]
 [3]
 [4]
 [5, 8]
 [6, 7, 117, 118, 119, 120, 121, 122, 123, 124  …  3160, 3169, 3202, 3203, 3211, 3212, 3228, 3235, 3290, 3296]
 [6, 7, 117, 118, 119, 120, 121, 122, 123, 124  …  3127, 3160, 3169, 3211, 3212, 3228, 3229, 3277, 3290, 3352]
 [5, 8]
 [9]
 [10]
 [11]
 [12]
 [13]
 ⋮
 [1443, 1764, 1828, 2292, 2294, 2490, 2696, 3192, 3378]
 [1163, 1451, 1983, 1984, 1995, 2642, 2672, 3134, 3346, 3362, 3379]
 [113, 114, 115, 116, 117, 130, 131, 172, 856, 1867, 1887, 1970, 1972, 2050, 3089, 3124, 3226, 3235, 3319, 3380]
 [111, 183, 1854, 1865, 1891, 2008, 2136, 2137, 3285, 3302, 3381]
 [163, 1344, 1500, 2106, 2109, 2514, 2852, 3005, 3076, 3210, 3382]
 [1902, 1907, 1910, 2078, 2079, 2767, 2929, 3383]
 [1253, 1357, 1361, 1367, 1468, 1892, 1893, 2088, 2419, 2982, 3184, 3384]
 [206, 1718, 1856, 2009, 2010, 2478, 3093, 3161, 3385]
 [450, 452, 1720, 2985, 3227, 3314, 3328, 3386]
 [169, 1687, 1700, 1860, 2012, 2013, 3106, 3120, 3171, 3387]
 [3388]
 [3389]

In [17]:
function nonLocalGk(G_k_prev_x,G_k_prev_y)
    GkVec_x = evaluate(G_k_prev_x,x_S)
    GkVec_y = evaluate(G_k_prev_y,x_S)

    caches_x = [array_cache(GkVec_x) for k in 1:Threads.nthreads()]
    caches_y = [array_cache(GkVec_y) for k in 1:Threads.nthreads()]

    # Pre-allocate arrays
    Gk_nds_x = zeros(Nₙ)
    Gk_nds_y = zeros(Nₙ)
    Gk_sum_x = zeros(Nₙ)
    Gk_sum_y = zeros(Nₙ)
    Gk_count_x = zeros(Int, Nₙ)
    Gk_count_y = zeros(Int, Nₙ)
    Threads.@threads for iel in 1:Nₑ
        cache_x = caches_x[Threads.threadid()]
        cache_y = caches_y[Threads.threadid()]
        ElNdInd = elem[iel]
    val_G_x = getindex!(cache_x, GkVec_x, iel)
    val_G_y = getindex!(cache_y, GkVec_y, iel)
for node in ElNdInd
        Gk_sum_x[node] += val_G_x[1]
        Gk_sum_y[node] += val_G_y[1]
        Gk_count_x[node] += 1
        Gk_count_y[node] += 1
        end
    end
Gk_nds_x = Gk_sum_x ./ Gk_count_x
Gk_nds_y = Gk_sum_y ./ Gk_count_y
  Gk_nds_NL_x = zeros(Nₙ)
  Gk_nds_NL_y = zeros(Nₙ)
    Threads.@threads for nd_id in 1:Nₙ
        NeighHood = idxs[nd_id]
        @inbounds Gk_nds_NL_x[nd_id] = (sum(Gk_nds_x[NeighHood]) ) / (length(NeighHood))
        @inbounds Gk_nds_NL_y[nd_id] = (sum(Gk_nds_y[NeighHood]) ) / (length(NeighHood))
    end
    return Gk_nds_x,Gk_nds_y
end

nonLocalGk (generic function with 1 method)

In [18]:
LoadTagId = get_tag_from_name(labels,"BottomEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

GenericCellField():
 num_cells: 10
 DomainStyle: ReferenceDomain()
 Triangulation: BoundaryTriangulation()
 Triangulation id: 2466160940788321471

In [19]:
reffe_Disp = ReferenceFE(lagrangian,VectorValue{2,Float64},2)
V0_Disp = TestFESpace(model,reffe_Disp;
          conformity=:H1,
          dirichlet_tags=["BottomEdge","TopEdge"],
          dirichlet_masks=[(true,true),(false,true)])


UnconstrainedFESpace()

In [20]:
reffephi  = ReferenceFE(lagrangian,Float64,1)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)

UnconstrainedFESpace()

In [21]:
function new_EnergyState(ψPlusPrev_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
        damaged = false
    else
        ψPlus_out = ψPlus_in
        damaged = true
    end
    damaged, ψPlus_out   
end

new_EnergyState (generic function with 2 methods)

In [22]:
function step_disp(uApp,uh,m_p_x,m_p_y,σ_p_x,σ_p_y,ϕ,G_k_cell_x,G_k_cell_y)
uApp1(x) = VectorValue(0.0,0.0)
uApp2(x) = VectorValue(0.0,uApp)  
U_Disp = TrialFESpace(V0_Disp,[uApp1,uApp2])
σ_eq_s_x = σ_eq1∘((ε(uh))*ϕ)
σ_eq_s_y = σ_eq2∘((ε(uh))*ϕ)
G_k_in_x,G_k_in_y = G_kill(σ_eq_s_x,σ_eq_s_y)
update_state!(new_EnergyState,G_k_cell_x,G_k_in_x)
update_state!(new_EnergyState,G_k_cell_y,G_k_in_y)
G_k_nds_x,G_k_nds_y = nonLocalGk(G_k_cell_x,G_k_cell_y)
ϕ,m_p_x,σ_p_x,m_p_y,σ_p_y = barw_closed_form(delT,m_p_x,m_p_y,G_k_nds_x,G_k_nds_y,σ_p_x,σ_p_y)
a(u,v) = ∫( ε(v) ⊙ ((σ∘((ε(u))))  .*(ϕ)))dΩ
l(v) = 0.0
op = AffineFEOperator(a,l,U_Disp,V0_Disp)
ls = LUSolver()
solver = LinearFESolver(ls)
uh = solve(solver,op)
    return uh, ϕ, m_p_x,m_p_y,σ_p_x,σ_p_y,G_k_cell_x,G_k_cell_y  
end

step_disp (generic function with 1 method)

In [23]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

CellPoint()

In [24]:
m_p_x = zeros(Nₙ) .+ 1e-6 
σ_p_x = zeros(Nₙ) .+ 1e-4 
m_p_y = zeros(Nₙ) .+ 1e-6 
σ_p_y = zeros(Nₙ) .+ 1e-4 

tol = 1e-1
target = (0.50, 1.0) 
row = findfirst(n -> norm((n[1]-target[1], n[2]-target[2])) < tol, nodes)
m_p_ct_x = Float64[]
G_k_nds_y = zeros(Nₙ)
σ_p_ct_x = Float64[]
m_p_ct_y = Float64[]
σ_p_ct_y = Float64[]
push!(m_p_ct_x, m_p_x[row])
push!(σ_p_ct_x, σ_p_x[row])
push!(m_p_ct_y, m_p_y[row])
push!(σ_p_ct_y, σ_p_y[row])

1-element Vector{Float64}:
 0.0001

In [25]:
cd("JKOWithHessian-checkanisotropy")

In [26]:
uh = zero(V0_Disp)
Tmax = 0.075 
delT = 2e-4
vApp = 1.0
count_n = 0
T = 0.0
Load = Float64[]
Displacement = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)

uh_prev = zero(V0_Disp)
uh_in_FE = uh
f_new = 1.0
ϕ_prev = interpolate_everywhere(f_new,Vphi)
ϕ = interpolate_everywhere(f_new,Vphi)
innerMax = 10
G_k_cell_x = CellState(0.0,dΩ_ro)
G_k_cell_y = CellState(0.0,dΩ_ro)
σ_eq_s_x= σ_eq1∘(ε(uh))
σ_eq_s_y= σ_eq2∘(ε(uh))

start_time = time()
while T <= Tmax
    count_n = count_n + 1
T = T + delT
uApp  = T*vApp
    print("\n Entering displacemtent step :", float(uApp))
 for inner = 1:innerMax
uh,ϕ,m_p_x,m_p_y,σ_p_x,σ_p_y,G_k_cell_x,G_k_cell_y =  step_disp(uApp,uh,m_p_x,m_p_y,σ_p_x,σ_p_y,ϕ,G_k_cell_x,G_k_cell_y)
e = uh - uh_in_FE
err = sqrt(sum( ∫( e⊙e )*dΩ ))
ϕ_prev = ϕ
uh_in_FE = uh
print("\n error = ",float(err))
        if err < 1e-8
            break 
        end  
    end
Node_Force = sum(∫( n_Γ_Load ⋅ (σ∘( (ε(uh))) ).*ϕ)dΓ_Load)
push!(Load, abs(Node_Force[2]))
push!(Displacement, uApp)
push!(m_p_ct_x, m_p_x[row])
push!(σ_p_ct_x, σ_p_x[row])
push!(m_p_ct_y, m_p_y[row])
push!(σ_p_ct_y, σ_p_y[row])

 if mod(count_n,5) == 0
writevtk(Ω,"SENT_15degneg_results_$count_n",cellfields=["disp"=>uh,"phi"=>ϕ,"mean_x" => FEFunction(Vphi, m_p_x), "var_x" => FEFunction(Vphi, σ_p_x),"mean_y" => FEFunction(Vphi, m_p_y), "var_y" => FEFunction(Vphi, σ_p_y),"var_y" => FEFunction(Vphi, σ_p_y),"G_k_y" => σ_eq2∘((ε(uh))*ϕ),"G_k_x" => σ_eq1∘((ε(uh))*ϕ)  ])   
    end
    end
end_time = time()
elapsed_time = end_time - start_time


 Entering displacemtent step :0.0002
 error = 0.00018042326483833817
 error = 4.2575750970900885e-18
 Entering displacemtent step :0.0004
 error = 0.00018042326483837706
 error = 1.4958205822859846e-17
 Entering displacemtent step :0.0006000000000000001
 error = 0.00018042326483834202
 error = 9.004888785183117e-17
 Entering displacemtent step :0.0008
 error = 0.00018042326483836977
 error = 4.024917824215666e-17
 Entering displacemtent step :0.001
 error = 0.00018042326483830944
 error = 7.432416754065777e-17
 Entering displacemtent step :0.0012000000000000001
 error = 0.00018042326483843894
 error = 4.696537753057334e-16
 Entering displacemtent step :0.0014000000000000002
 error = 0.00018042326483839043
 error = 1.0138213631270825e-16
 Entering displacemtent step :0.0016000000000000003
 error = 0.00018042326483832066
 error = 4.793707244932937e-17
 Entering displacemtent step :0.0018000000000000004
 error = 0.0001804232648383384
 error = 5.027921242730567e-16
 Entering displacemtent

2469.5490000247955